# SQL in Python – Mit PostgreSQL verbinden und Daten holen

*(Deutsche Übersetzung des Notebooks `03_fetching_the_data_eda.ipynb`. Nur die Texte und Kommentare sind übersetzt, der Code ist unverändert.)*

Du hast bereits gelernt, wie man sich mit einem SQL-Programm wie DBeaver mit einer Datenbank verbindet. DBeaver kann aber noch mehr: Du kannst damit SQL-Abfragen ausführen, neue Tabellen anlegen, sie mit Daten füllen und Daten abrufen.

Auch Python kann SQL-Abfragen ausführen und das Ergebnis in ein Python-Objekt holen, zum Beispiel in einen pandas-DataFrame. Statt eine .csv-Datei aus DBeaver zu exportieren, kannst du die Daten direkt in Python holen und dort weiterarbeiten. So sparst du Schritte, weil du dich direkt aus Python mit der Datenbank verbindest und kein extra SQL-Programm brauchst.

Wenn du die Daten in Python in der richtigen Form hast, kannst du sie in eine .csv-Datei exportieren. Diese Datei ist nur für dich selbst gedacht. Bitte verschicke keine .csv-Dateien. Die Datenbank ist immer die maßgebliche Quelle für die Daten.

Eine Kopie als .csv-Datei (oder in einem anderen Format) kann deine Analyse aber beschleunigen. Stell dir vor, die Abfrage dauert 25 Minuten. Wenn du in deinem Python-Code einen Fehler gemacht hast, musst du vielleicht zu den Originaldaten zurück. Statt die SQL-Abfrage noch einmal laufen zu lassen und zu warten, liest du einfach die .csv-Datei von deiner Festplatte in Python ein und machst mit der Analyse weiter.

**In diesem Notebook siehst du 2 Wege, dich mit SQL-Datenbanken zu verbinden und die Daten in eine CSV-Datei zu exportieren.**


## Mit Python eine Verbindung zu einer PostgreSQL-Datenbank herstellen

Es gibt 2 Python-Pakete, die man üblicherweise benutzt, um sich mit SQL-Datenbanken zu verbinden: `psycopg2` und `sqlalchemy`.

### Verbinden mit psycopg2

In [ ]:
import pandas as pd
import psycopg2


Um eine Verbindung zu unserer PostgreSQL-Datenbank herzustellen, brauchen wir folgende Informationen:

- host = die Adresse des Rechners, auf dem die Datenbank liegt
- port = die Nummer des „Tors“, über das die Verbindung erlaubt ist
- database = der Name der Datenbank
- user = der Benutzername
- password = das Passwort des Benutzers

Weil diese Datenbank-Informationen nicht auf GitHub veröffentlicht werden sollen, schreiben wir sie in eine `.env`-Datei. Diese Datei steht in der `.gitignore`.
In solchen Dateien kannst du Informationen speichern, die nicht veröffentlicht werden sollen.
Mit dem Paket `dotenv` kannst du `.env`-Dateien lesen und die Werte daraus holen.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

DATABASE = os.getenv("DATABASE")
USER_DB = os.getenv("USER_DB")
PASSWORD = os.getenv("PASSWORD")
HOST = os.getenv("HOST")
PORT = os.getenv("PORT")


In [ ]:
USER_DB
print(USER_DB)

In [ ]:
HOST
print(HOST)

In [ ]:
PASSWORD
print(PASSWORD)

In [ ]:
DATABASE
print(DATABASE)

In [ ]:
pwd

Die Funktion aus dem Paket psycopg2, die eine Verbindung herstellt, heißt `connect()`.
`connect()` braucht die oben genannten Angaben, um sich mit der Datenbank zu verbinden.

In [ ]:
# Verbindungsobjekt conn erstellen
conn = psycopg2.connect(
    database="postgres", user="aieng_de_210926_neuronale_nelken", password="mialovesicecream", host="ds-sql-playground.c8g8r1deus2v.eu-central-1.rds.amazonaws.com", port="5432"
)

### Daten mit psycopg2 aus der Datenbank holen

Bevor wir mit unserer Verbindung Daten holen können, müssen wir einen **Cursor** erstellen. Mit einem Cursor kann Python-Code PostgreSQL-Befehle in der Datenbank ausführen.
Den Cursor erstellt man mit der Methode `cursor()` unseres Verbindungsobjekts `conn`.

In [ ]:
cur = conn.cursor()

Jetzt können wir SQL-Abfragen mit `cur.execute('ABFRAGE')` ausführen und dann mit `cur.fetchall()` die Daten holen:

In [ ]:
cur.execute("SELECT * FROM eda.king_county_house_sales LIMIT 10")
cur.fetchall()

Mit `conn.close()` schließt du die Verbindung wieder.

In [ ]:
# Verbindung schließen
conn.close()

Wir wollen aber mit den Daten arbeiten. Am einfachsten ist es, die Daten in pandas-DataFrames zu laden. Dafür gibt es `pd.read_sql_query` oder `pd.read_sql_table` oder, am bequemsten, `pd.read_sql`.

`pd.read_sql` ist eine praktische Hülle um `read_sql_table` und `read_sql_query` (damit älterer Code weiter funktioniert). Je nachdem, was du übergibst, leitet sie an die passende Funktion weiter: Eine SQL-Abfrage geht an `read_sql_query`, ein Tabellenname geht an `read_sql_table`. Die jeweilige Funktion kann noch genauere Hinweise haben, die hier nicht aufgeführt sind.

In [ ]:
# Verbindung wieder öffnen, weil wir sie geschlossen haben
conn = psycopg2.connect(
    database=DATABASE, user=USER_DB, password=PASSWORD, host=HOST, port=PORT
)

In [ ]:
# Blockierte Verbindung zurücksetzen
conn.rollback()

# Abfrage: beide Tabellen verbinden
query_string = """
SELECT s.date, s.price, d.*
FROM eda.king_county_house_details AS d
INNER JOIN eda.king_county_house_sales AS s
    ON d.id = s.house_id;
"""

# Daten in einen pandas-DataFrame laden
df_psycopg = pd.read_sql(query_string, conn)

In [ ]:
df_psycopg.head()

In [ ]:
# Verbindung schließen
conn.close()

In [ ]:
query_string = """
SELECT * 
FROM king_county_house_details det
FULL JOIN king_county_house_sales sal 
ON det.id = sal.house_id;
"""

In [ ]:
query_string = """
SELECT s.date, s.price, d.*
FROM eda.king_county_house_details AS d
INNER JOIN eda.king_county_house_sales AS s
    ON d.id = s.house_id;
    """

In [ ]:
# Kontrolle: Anzahl Zeilen und Spalten
df_psycopg.shape

In [ ]:
# -->SKELETT: alles Funktionen noch ohne Inhalt

# 1. Laden
def daten_laden():
    pass

# 2. Anschauen
def daten_anschauen(df):
    pass

# 3. Bereinigen
def daten_bereinigen(df):
    pass

# 4. Fragen stellen: neue Spalten und Grenzen
def entfernung_berechnen(df):
    pass

def preisgrenzen_berechnen(df):
    pass

# 5. Beantworten
def hypothese_1_pruefen(df):
    pass

def hypothese_2_pruefen(df):
    pass

def hypothese_3_pruefen(df, untere_grenze, obere_grenze):
    pass

# 6. Zeigen
def diagramm_h1(df):
    pass

def diagramm_h2(df):
    pass

def diagramm_h3(df, untere_grenze, obere_grenze):
    pass

In [ ]:
df_psycopg.head()

In [ ]:
# Daten in eine CSV-Datei exportieren
df_psycopg.to_csv("data/eda.csv", index=False)

### Verbinden und Daten holen mit SQLAlchemy

`sqlalchemy` funktioniert ähnlich. Hier erstellst du eine **Engine** mit dem Datenbank-String (ein Link, der alle Informationen enthält, die wir oben in das `conn`-Objekt eingetragen haben).

In [ ]:
from sqlalchemy import create_engine

# Datenbank-String aus der .env-Datei lesen
load_dotenv()

DB_STRING = os.getenv("DB_STRING")

if DB_STRING is None:
    raise ValueError("DB_STRING is not set in the environment.")

db = create_engine(DB_STRING)

Dann kannst du die Engine zusammen mit einer Abfrage benutzen, um die Daten in einen pandas-DataFrame zu laden.

In [ ]:
# Daten in einen pandas-DataFrame laden
query_string = "SELECT * FROM eda.king_county_house_sales"
df_sqlalchemy = pd.read_sql(query_string, db)

In [ ]:
df_sqlalchemy.head()

Weil wir die Abfragen nicht immer wieder ausführen wollen, können wir die Daten in eine .csv-Datei exportieren und sie so auch in anderen Notebooks benutzen.

In [ ]:
# Daten in eine CSV-Datei exportieren
df_sqlalchemy.to_csv("data/eda.csv", index=False)

In [ ]:
# Daten aus einer CSV-Datei laden
df_import = pd.read_csv("data/eda.csv")